# WebTable Target Column Preview

Use this notebook to inspect raw CSV tables and GT-related columns from the LakeBench WebTable Join data.

The notebook reads the prepared DeepJoin failure-analysis artifacts, so it does not scan the full WebTable directory.

In [ ]:
from pathlib import Path
import csv
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)

REPO_ROOT = Path("/home/syback/AIST/dynamic-join-index")
MANIFEST_PATH = REPO_ROOT / "sybaek/failure_analysis/deepjoin/artifacts/column_manifest.csv"
GT_PAIRS_PATH = REPO_ROOT / "sybaek/failure_analysis/deepjoin/artifacts/ground_truth_pairs.csv"
FAILURE_TABLE_PATH = REPO_ROOT / "sybaek/failure_analysis/deepjoin/results/failure_table.csv"

print("repo:", REPO_ROOT)
print("manifest exists:", MANIFEST_PATH.exists())
print("gt pairs exists:", GT_PAIRS_PATH.exists())
print("failure table exists:", FAILURE_TABLE_PATH.exists())

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")
gt_pairs = pd.read_csv(GT_PAIRS_PATH, dtype=str).fillna("")

failure = None
if FAILURE_TABLE_PATH.exists():
    failure = pd.read_csv(FAILURE_TABLE_PATH, dtype=str).fillna("")
    for col in ["exact_rank", "query_containment", "candidate_containment", "jaccard", "max_containment"]:
        if col in failure.columns:
            failure[col] = pd.to_numeric(failure[col], errors="coerce")

print("GT-related unique columns:", len(manifest))
print("GT pairs:", len(gt_pairs))
if failure is not None:
    print("failure rows:", len(failure))

manifest.head()

## Query 1 GT Pair Example

`Query 1` means the first unique `query_column_id` in `ground_truth_pairs.csv`. This section shows the query column, one GT-paired candidate column, their raw tables, and the exact equi-join result.

In [ ]:
column_lookup = manifest.set_index("column_id", drop=False)

def resolve_table_path(table_path):
    path = Path(table_path)
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path

def get_column_row(column_id):
    if column_id not in column_lookup.index:
        raise KeyError(f"column_id not found: {column_id}")
    return column_lookup.loc[column_id]

def load_raw_table_for_column(column_id, nrows=None):
    row = get_column_row(column_id)
    path = resolve_table_path(row["table_path"])
    return pd.read_csv(path, dtype=str, nrows=nrows, encoding="utf-8-sig", on_bad_lines="skip").fillna("")

def column_info(column_id):
    row = get_column_row(column_id)
    return {
        "column_id": row["column_id"],
        "table_file": Path(row["table_path"]).name,
        "table_path": str(resolve_table_path(row["table_path"])),
        "column_name": row["column_name"],
    }

def get_query_pairs(query_number=1):
    query_ids = gt_pairs["query_column_id"].drop_duplicates().tolist()
    if query_number < 1 or query_number > len(query_ids):
        raise IndexError(f"query_number must be between 1 and {len(query_ids)}")
    query_id = query_ids[query_number - 1]
    pairs = gt_pairs.loc[gt_pairs["query_column_id"] == query_id].copy().reset_index(drop=True)
    pairs.insert(0, "candidate_number", range(1, len(pairs) + 1))
    pairs["query_column_name"] = pairs["query_column_id"].map(lambda x: column_info(x)["column_name"])
    pairs["candidate_column_name"] = pairs["candidate_column_id"].map(lambda x: column_info(x)["column_name"])
    pairs["candidate_table_file"] = pairs["candidate_column_id"].map(lambda x: column_info(x)["table_file"])
    if failure is not None:
        keep = ["pair_id", "exact_rank", "rank_group", "query_containment", "candidate_containment", "jaccard", "max_containment"]
        keep = [c for c in keep if c in failure.columns]
        pairs = pairs.merge(failure[keep], on="pair_id", how="left")
    return query_id, pairs

query_1_id, query_1_pairs = get_query_pairs(query_number=1)
print("Query 1 column:")
print(column_info(query_1_id))
print("Number of GT candidate pairs for Query 1:", len(query_1_pairs))
query_1_pairs

In [ ]:
def normalized_values(series):
    values = series.dropna().astype(str).str.strip()
    return values[values != ""]

def exact_join_preview(query_number=1, candidate_number=1, table_preview_rows=10, value_preview_rows=30, join_preview_rows=50, join_max_rows=None):
    query_id, pairs = get_query_pairs(query_number=query_number)
    if candidate_number < 1 or candidate_number > len(pairs):
        raise IndexError(f"candidate_number must be between 1 and {len(pairs)}")

    pair = pairs.iloc[candidate_number - 1]
    candidate_id = pair["candidate_column_id"]
    qinfo = column_info(query_id)
    cinfo = column_info(candidate_id)

    qdf = load_raw_table_for_column(query_id, nrows=join_max_rows)
    cdf = load_raw_table_for_column(candidate_id, nrows=join_max_rows)
    qcol = qinfo["column_name"]
    ccol = cinfo["column_name"]

    print("GT pair:", pair["pair_id"])
    print("Query column:", qinfo)
    print("Candidate column:", cinfo)
    print("\nPrecomputed DeepJoin/overlap metrics for this GT pair")
    metric_cols = [
        "exact_rank", "rank_group", "query_containment", "candidate_containment",
        "jaccard", "max_containment"
    ]
    metric_row = {col: pair.get(col, "") for col in metric_cols if col in pair.index}
    display(pd.DataFrame([metric_row]))

    q_values = normalized_values(qdf[qcol]) if qcol in qdf.columns else pd.Series([], dtype=str)
    c_values = normalized_values(cdf[ccol]) if ccol in cdf.columns else pd.Series([], dtype=str)
    q_set = set(q_values)
    c_set = set(c_values)
    overlap = q_set & c_set
    metrics = pd.DataFrame([{
        "query_rows_loaded": len(qdf),
        "candidate_rows_loaded": len(cdf),
        "query_distinct_values": len(q_set),
        "candidate_distinct_values": len(c_set),
        "overlap_distinct_values": len(overlap),
        "query_containment": len(overlap) / len(q_set) if q_set else 0,
        "candidate_containment": len(overlap) / len(c_set) if c_set else 0,
        "jaccard": len(overlap) / len(q_set | c_set) if (q_set or c_set) else 0,
    }])

    qwork = qdf.copy()
    cwork = cdf.copy()
    qwork["__join_key__"] = qwork[qcol].astype(str).str.strip()
    cwork["__join_key__"] = cwork[ccol].astype(str).str.strip()
    qwork = qwork[qwork["__join_key__"] != ""]
    cwork = cwork[cwork["__join_key__"] != ""]
    joined = qwork.merge(cwork, on="__join_key__", how="inner", suffixes=("__query", "__candidate"))

    side_by_side_values = pd.concat([
        q_values.head(value_preview_rows).reset_index(drop=True).rename(f"query::{qcol}"),
        c_values.head(value_preview_rows).reset_index(drop=True).rename(f"candidate::{ccol}"),
    ], axis=1)

    print("\nQuery table preview")
    display(qdf.head(table_preview_rows))
    print("\nCandidate table preview")
    display(cdf.head(table_preview_rows))
    print("\nQuery/Candidate column values")
    display(side_by_side_values)
    print("\nJoinability metrics from loaded rows")
    display(metrics)
    print(f"\nExact equi-join result: {len(joined)} rows")
    display(joined.head(join_preview_rows))

    return {
        "pair": pair,
        "query_info": qinfo,
        "candidate_info": cinfo,
        "query_table": qdf,
        "candidate_table": cdf,
        "metrics": metrics,
        "joined": joined,
    }

# Query 1, first GT-paired candidate column.
query_1_example = exact_join_preview(query_number=1, candidate_number=1)

## Search GT-Related Columns

Search by column name, table file name, or full `column_id`.

In [ ]:
def search_columns(keyword, limit=30):
    keyword = str(keyword).lower()
    mask = (
        manifest["column_id"].str.lower().str.contains(keyword, regex=False)
        | manifest["column_name"].str.lower().str.contains(keyword, regex=False)
        | manifest["table_path"].str.lower().str.contains(keyword, regex=False)
    )
    return manifest.loc[mask].head(limit)

# Example: search_columns("Player")
search_columns("Player", limit=10)

## Preview One Raw Column

Use a `column_id` from `manifest` or search results.

In [ ]:
def resolve_table_path(table_path):
    path = Path(table_path)
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path

def load_table_head(table_path, nrows=20):
    path = resolve_table_path(table_path)
    return pd.read_csv(path, dtype=str, nrows=nrows, encoding="utf-8-sig", on_bad_lines="skip").fillna("")

def get_column_row(column_id):
    rows = manifest.loc[manifest["column_id"] == column_id]
    if rows.empty:
        raise KeyError(f"column_id not found: {column_id}")
    return rows.iloc[0]

def preview_column(column_id, nrows=30):
    row = get_column_row(column_id)
    table_path = resolve_table_path(row["table_path"])
    df = load_table_head(table_path, nrows=nrows)
    col = row["column_name"]
    print("column_id:", row["column_id"])
    print("table_path:", table_path)
    print("column_name:", col)
    print("table_shape_preview:", df.shape)
    if col not in df.columns:
        print("column not found in pandas columns. Available columns:")
        print(list(df.columns))
        return df
    return df[[col]].head(nrows)

example_column_id = manifest.iloc[0]["column_id"]
preview_column(example_column_id, nrows=20)

## Preview Full Table Head

This shows the first rows of the raw CSV table containing a target column.

In [ ]:
def preview_table_for_column(column_id, nrows=20):
    row = get_column_row(column_id)
    table_path = resolve_table_path(row["table_path"])
    print("column_id:", row["column_id"])
    print("table_path:", table_path)
    print("target_column:", row["column_name"])
    return load_table_head(table_path, nrows=nrows)

preview_table_for_column(example_column_id, nrows=10)

## Preview One GT Pair

This prints the query/candidate raw paths and displays their cell values side by side.

In [ ]:
def column_values(column_id, nrows=30):
    row = get_column_row(column_id)
    df = load_table_head(row["table_path"], nrows=nrows)
    col = row["column_name"]
    if col not in df.columns:
        return pd.Series([], dtype=str, name=col)
    return df[col].astype(str).reset_index(drop=True).rename(col)

def preview_gt_pair(pair_id=None, row_index=0, nrows=30):
    if pair_id is not None:
        rows = gt_pairs.loc[gt_pairs["pair_id"] == pair_id]
        if rows.empty:
            raise KeyError(f"pair_id not found: {pair_id}")
        pair = rows.iloc[0]
    else:
        pair = gt_pairs.iloc[row_index]

    qid = pair["query_column_id"]
    cid = pair["candidate_column_id"]
    qrow = get_column_row(qid)
    crow = get_column_row(cid)

    print("pair_id:", pair["pair_id"])
    print("query_column_id:", qid)
    print("query_path:", resolve_table_path(qrow["table_path"]))
    print("candidate_column_id:", cid)
    print("candidate_path:", resolve_table_path(crow["table_path"]))

    qvals = column_values(qid, nrows=nrows).rename("query_values")
    cvals = column_values(cid, nrows=nrows).rename("candidate_values")
    return pd.concat([qvals, cvals], axis=1)

preview_gt_pair(row_index=0, nrows=20)

## Inspect Failure Cases

If `failure_table.csv` exists, use DeepJoin rank and overlap features to select interesting GT pairs.

In [ ]:
if failure is not None:
    display_cols = [
        "pair_id", "query_column_id", "candidate_column_id", "exact_rank",
        "rank_group", "query_containment", "candidate_containment", "jaccard", "max_containment"
    ]
    existing = [c for c in display_cols if c in failure.columns]
    hard_cases = failure.sort_values("exact_rank", ascending=False)[existing].head(20)
    display(hard_cases)
else:
    print("failure_table.csv not found")

In [ ]:
def preview_failure_pair(row_index=0, nrows=30, sort_by="exact_rank", ascending=False):
    if failure is None:
        raise FileNotFoundError(FAILURE_TABLE_PATH)
    row = failure.sort_values(sort_by, ascending=ascending).iloc[row_index]
    print("exact_rank:", row.get("exact_rank", ""))
    print("rank_group:", row.get("rank_group", ""))
    print("query_containment:", row.get("query_containment", ""))
    print("candidate_containment:", row.get("candidate_containment", ""))
    print("jaccard:", row.get("jaccard", ""))
    return preview_gt_pair(pair_id=row["pair_id"], nrows=nrows)

# Example: the hardest pair by exact rank
preview_failure_pair(row_index=0, nrows=20)